In [26]:
# Why this is needed:
import itertools
vocab_sizes = [100, 250, 500, 750, 1000]
num_els = [3,4, 5]
for vocab_size, num_el in itertools.product(vocab_sizes, num_els):
    size = vocab_size ** num_el
    print(f"Vocab size: {vocab_size}, Num elements: {num_el}")
    print(f"Total size: {size} entries (~{(size * 4) / (1024 ** 3):.2f} GB as float32)")
    print()


Vocab size: 100, Num elements: 3
Total size: 1000000 entries (~0.00 GB as float32)

Vocab size: 100, Num elements: 4
Total size: 100000000 entries (~0.37 GB as float32)

Vocab size: 100, Num elements: 5
Total size: 10000000000 entries (~37.25 GB as float32)

Vocab size: 250, Num elements: 3
Total size: 15625000 entries (~0.06 GB as float32)

Vocab size: 250, Num elements: 4
Total size: 3906250000 entries (~14.55 GB as float32)

Vocab size: 250, Num elements: 5
Total size: 976562500000 entries (~3637.98 GB as float32)

Vocab size: 500, Num elements: 3
Total size: 125000000 entries (~0.47 GB as float32)

Vocab size: 500, Num elements: 4
Total size: 62500000000 entries (~232.83 GB as float32)

Vocab size: 500, Num elements: 5
Total size: 31250000000000 entries (~116415.32 GB as float32)

Vocab size: 750, Num elements: 3
Total size: 421875000 entries (~1.57 GB as float32)

Vocab size: 750, Num elements: 4
Total size: 316406250000 entries (~1178.71 GB as float32)

Vocab size: 750, Num eleme

In [7]:
import torch
import pickle
import tensorflow as tf
import tensorly as tl
from tensorly.decomposition import tucker, non_negative_tucker, non_negative_tucker_hals
import time
from typing import List, Tuple
from utils import DATA_DIR, select_gpu

select_gpu()
# we load the tensor and vocabularies
# we save the respective vocabularies as well
# vocab = {
#     "vocab_v": vocab_v,
#     "vocab_s": vocab_s,
#     "vocab_o": vocab_o,
#     "v2i": v2i,
#     "s2i": s2i,
#     "o2i": o2i
# }
with open(DATA_DIR/"tensors/fineweb_old/vocabularies/1000.pkl", "rb") as f:
    vocab = pickle.load(f)
vocab_v = vocab["vocab_v"]
vocab_s = vocab["vocab_s"]
vocab_o = vocab["vocab_o"]
v2i = vocab["v2i"]
s2i = vocab["s2i"]
o2i = vocab["o2i"]

def _torch_or_tucker_load(path, map_location="cpu"):
    """Tries to load a torch-saved file, if fails, tries pickle."""
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except RuntimeError:
        with open(path, "rb") as f:
            return pickle.load(f)

class TupleTensor:
    def __init__(self, path: str):
        self.tensor = _torch_or_tucker_load(path)


    def sparse_representation(self):
        # we return the sparse representation of the tensor
        # we check if our tensor is a tensorflow tensor or make it one
        if not isinstance(self.tensor, tf.Tensor):
            self.tensor = tf.convert_to_tensor(self.tensor)
        return tf.sparse.from_dense(self.tensor)

    def tensor_to_sparse(self):
        self.tensor = self.sparse_representation()

    def decompose(self, non_negative=True, hals=True, rank=None,
                  n_iter_max=100, init="random", tol=1e-12, verbose=False):
        if rank is None:
            rank = [100, 100, 100]
        tic = time.time()

        if not non_negative:
            decomp = tucker
        elif hals:
            decomp = non_negative_tucker_hals
        else:
            decomp = non_negative_tucker

        core, factors = decomp(self.tensor, rank=rank, n_iter_max=n_iter_max,
                               init=init, tol=tol, verbose=verbose)
        toc = time.time()
        print("Time taken for decomposition", toc-tic)
        return core, factors

tensor = TupleTensor(DATA_DIR/"tensors/fineweb_old/populated/sii_100.pt")


Selected GPU 1 with 451.75 MB used memory.


In [3]:
from tensorly.contrib.sparse.decomposition import non_negative_tucker as nnt_sparse
from tensorly.contrib.sparse import tensor as sparse_tensorly
import numpy as np
import sparse


In [4]:
tl.set_backend("numpy")

In [5]:
sparse_tensor_tensorly = sparse_tensorly(tensor.tensor)

In [8]:
result = nnt_sparse(sparse_tensor_tensorly, init="random", rank=[100,100,100], n_iter_max=100, verbose=True, tol=1e-10)

reconstruction error=0.9884003400802612, variation=7.641315460205078e-05.
reconstruction error=0.9883901476860046, variation=1.0192394256591797e-05.
reconstruction error=0.9883840680122375, variation=6.079673767089844e-06.
reconstruction error=0.9883775115013123, variation=6.556510925292969e-06.
reconstruction error=0.9883695244789124, variation=7.987022399902344e-06.
reconstruction error=0.9883596897125244, variation=9.834766387939453e-06.
reconstruction error=0.9883474111557007, variation=1.2278556823730469e-05.
reconstruction error=0.9883314967155457, variation=1.5914440155029297e-05.
reconstruction error=0.9883105754852295, variation=2.092123031616211e-05.
reconstruction error=0.988282322883606, variation=2.8252601623535156e-05.
reconstruction error=0.9882432222366333, variation=3.910064697265625e-05.
reconstruction error=0.9881879091262817, variation=5.53131103515625e-05.
reconstruction error=0.9881079196929932, variation=7.998943328857422e-05.
reconstruction error=0.9879904389381

In [9]:
result

Decomposed rank-(100, 100, 100) TuckerTensor of shape (100, 100, 100) 

In [12]:
core, factors = result

In [19]:
core = core.todense()

In [20]:
factors = [f.todense() for f in factors]

AttributeError: 'numpy.ndarray' object has no attribute 'todense'

In [21]:
print(core, factors)

[[[0.00000000e+00 0.00000000e+00 1.91555258e-37 ... 0.00000000e+00
   0.00000000e+00 0.00000000e+00]
  [9.90470722e-02 1.73406085e-19 1.07540982e-23 ... 5.40239599e-29
   2.71947496e-18 0.00000000e+00]
  [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
   0.00000000e+00 6.45162633e-38]
  ...
  [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 1.12278391e-23
   0.00000000e+00 1.95051456e-07]
  [7.87085418e-35 1.67094611e-22 4.87877717e-29 ... 1.41182205e-29
   1.44150226e-32 4.13162355e-27]
  [1.89161664e-09 9.26701992e-38 3.19730453e-16 ... 1.03768626e-11
   2.53438637e-23 0.00000000e+00]]

 [[1.36363617e-33 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
   0.00000000e+00 5.35440880e-27]
  [7.18193570e-30 0.00000000e+00 0.00000000e+00 ... 1.78613598e-21
   5.45442647e-38 1.84868412e+01]
  [0.00000000e+00 0.00000000e+00 1.88255575e-36 ... 0.00000000e+00
   0.00000000e+00 1.00417048e-41]
  ...
  [4.15453491e+01 0.00000000e+00 8.58704353e-21 ... 1.11523010e-23
   1.36056

In [22]:
V = S = O = 1000
print(f"Total size: {V * S * O} entries (~{(V * S * O * 4) / (1024 ** 3):.2f} GB as float32)")

Total size: 1000000000 entries (~3.73 GB as float32)


Vocab size: 100, Num elements: 3
Total size: 1000000 entries (~0.00 GB as float32)

Vocab size: 100, Num elements: 4
Total size: 100000000 entries (~0.37 GB as float32)

Vocab size: 100, Num elements: 5
Total size: 10000000000 entries (~37.25 GB as float32)

Vocab size: 250, Num elements: 3
Total size: 15625000 entries (~0.06 GB as float32)

Vocab size: 250, Num elements: 4
Total size: 3906250000 entries (~14.55 GB as float32)

Vocab size: 250, Num elements: 5
Total size: 976562500000 entries (~3637.98 GB as float32)

Vocab size: 500, Num elements: 3
Total size: 125000000 entries (~0.47 GB as float32)

Vocab size: 500, Num elements: 4
Total size: 62500000000 entries (~232.83 GB as float32)

Vocab size: 500, Num elements: 5
Total size: 31250000000000 entries (~116415.32 GB as float32)

Vocab size: 750, Num elements: 3
Total size: 421875000 entries (~1.57 GB as float32)

Vocab size: 750, Num elements: 4
Total size: 316406250000 entries (~1178.71 GB as float32)

Vocab size: 750, Num eleme